In [ ]:


!pip install -q pandas numpy scikit-learn xgboost joblib openpyxl spacy textblob
!python -m spacy download en_core_web_sm -q

import os, re, warnings, joblib
import numpy as np
import pandas as pd
import spacy

warnings.filterwarnings("ignore")

from textblob import TextBlob
from difflib import SequenceMatcher

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except:
    XGBOOST_AVAILABLE = False

nlp = spacy.load("en_core_web_sm", disable=["parser"])



# 1. DATASET PATHS


DOMAIN_PATH = "/content/domain_reputation.xlsx"
KEYWORDS_PATH = "/content/flagged_keywords.xlsx"
JOB_POSTS_PATH = "/content/job_posts_final_last01.xlsx"
RECRUITER_PATH = "/content/recruiter_profiles_table.xlsx"
SCAM_REPORTS_PATH = "/content/scam_reports_last06.xlsx"

for path in [DOMAIN_PATH, KEYWORDS_PATH, JOB_POSTS_PATH, RECRUITER_PATH, SCAM_REPORTS_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

print("All files found.")



# 2. LOAD DATA


job_posts = pd.read_excel(JOB_POSTS_PATH)
recruiters = pd.read_excel(RECRUITER_PATH)
scam_reports = pd.read_excel(SCAM_REPORTS_PATH)
domain_rep = pd.read_excel(DOMAIN_PATH)
keywords = pd.read_excel(KEYWORDS_PATH)

def clean_columns(df):
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )
    return df

job_posts = clean_columns(job_posts)
recruiters = clean_columns(recruiters)
scam_reports = clean_columns(scam_reports)
domain_rep = clean_columns(domain_rep)
keywords = clean_columns(keywords)

print(job_posts.shape, recruiters.shape, scam_reports.shape, domain_rep.shape, keywords.shape)



# 3. HELPER FUNCTIONS


def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-zA-Z0-9₹$@.\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def lemmatize_text(text):
    doc = nlp(str(text)[:100000])
    return " ".join([
        token.lemma_
        for token in doc
        if not token.is_stop and not token.is_punct and token.lemma_.strip()
    ])

def extract_number(value):
    if pd.isna(value):
        return 0
    value = str(value).replace(",", "")
    nums = re.findall(r"\d+\.?\d*", value)
    return float(nums[0]) if nums else 0

def bool_to_int(x):
    if pd.isna(x):
        return 0
    x = str(x).lower().strip()
    return 1 if x in ["true", "yes", "1", "valid", "verified"] else 0

def risk_level(score):
    if score <= 30:
        return "LOW RISK"
    elif score <= 60:
        return "MEDIUM RISK"
    elif score <= 80:
        return "HIGH RISK"
    return "CONFIRMED SCAM"

def similarity(a, b):
    return SequenceMatcher(None, str(a).lower(), str(b).lower()).ratio()



# 4. MAIN DATA


df = job_posts.copy()

if "is_flagged" not in df.columns:
    raise ValueError("Target column is_flagged not found.")

# Data leakage prevention:
# is_flagged = target only
# scam_score is never used as input
df["is_flagged"] = df["is_flagged"].astype(int)

df["title_text"] = df["title"].apply(clean_text)
df["description_text"] = df["description"].apply(clean_text)
df["combined_text_raw"] = df["title_text"] + " " + df["description_text"]

print("Applying spaCy lemmatization...")
df["combined_text"] = df["combined_text_raw"].apply(lemmatize_text)

df["salary_numeric"] = df["salary"].apply(extract_number)
df["domain_key"] = df["domain_name"].astype(str).str.lower().str.strip()
df["company_key"] = df["company_name"].astype(str).str.lower().str.strip()



# 5. FLAGGED KEYWORD NLP FEATURES


keywords["keyword"] = keywords["keyword"].astype(str).str.lower().str.strip()
keywords["fraud_weight"] = pd.to_numeric(keywords["fraud_weight"], errors="coerce").fillna(0)
keyword_dict = dict(zip(keywords["keyword"], keywords["fraud_weight"]))

def keyword_features(text):
    score, count = 0, 0
    text = str(text).lower()
    for kw, wt in keyword_dict.items():
        if kw and kw in text:
            score += float(wt)
            count += 1
    return pd.Series([score, count])

df[["keyword_score", "keyword_matches"]] = df["combined_text_raw"].apply(keyword_features)
max_keyword_score = max(df["keyword_score"].max(), 1)
df["keyword_risk"] = df["keyword_score"] / max_keyword_score



# 6. ADVANCED NLP FEATURES


fraud_patterns = [
    "registration fee", "training fee", "security deposit", "pay first",
    "instant joining", "no interview", "earn daily", "whatsapp hr",
    "limited seats", "guaranteed job", "guaranteed placement",
    "work 1 hour", "easy income", "urgent hiring", "click link"
]

urgency_words = [
    "urgent", "immediate", "limited", "hurry", "instant",
    "today", "last date", "apply now", "limited seats"
]

def advanced_nlp_features(text):
    text = str(text).lower()

    fraud_phrase_count = sum(1 for p in fraud_patterns if p in text)
    urgency_count = sum(1 for w in urgency_words if w in text)

    blob = TextBlob(text)
    sentiment_polarity = blob.sentiment.polarity
    sentiment_subjectivity = blob.sentiment.subjectivity

    doc = nlp(text[:100000])
    org_count = sum(1 for ent in doc.ents if ent.label_ == "ORG")
    money_count = sum(1 for ent in doc.ents if ent.label_ == "MONEY")
    person_count = sum(1 for ent in doc.ents if ent.label_ == "PERSON")

    return pd.Series([
        fraud_phrase_count,
        urgency_count,
        sentiment_polarity,
        sentiment_subjectivity,
        org_count,
        money_count,
        person_count
    ])

df[[
    "fraud_phrase_count",
    "urgency_count",
    "sentiment_polarity",
    "sentiment_subjectivity",
    "org_entity_count",
    "money_entity_count",
    "person_entity_count"
]] = df["combined_text_raw"].apply(advanced_nlp_features)



# 7. DOMAIN FEATURES


domain_rep["domain_key"] = domain_rep["domain_name"].astype(str).str.lower().str.strip()

domain_features = domain_rep[[
    "domain_key", "domain_age_days", "ssl_valid", "trust_score", "blacklisted"
]].drop_duplicates("domain_key")

df = df.merge(domain_features, on="domain_key", how="left")

df["domain_age_days"] = pd.to_numeric(df["domain_age_days"], errors="coerce").fillna(0)
df["ssl_valid"] = df["ssl_valid"].apply(bool_to_int)
df["trust_score"] = pd.to_numeric(df["trust_score"], errors="coerce").fillna(0.5)
df["blacklisted"] = df["blacklisted"].apply(bool_to_int)

df["domain_risk"] = (
    (1 - df["trust_score"].clip(0, 1)) * 0.50 +
    (1 - df["ssl_valid"]) * 0.20 +
    df["blacklisted"] * 0.20 +
    (df["domain_age_days"] < 90).astype(int) * 0.10
)

df["domain_company_similarity"] = df.apply(
    lambda row: similarity(row["company_key"], row["domain_key"]), axis=1
)

df["domain_impersonation_risk"] = (
    (df["domain_company_similarity"] < 0.25).astype(int)
)



# 8. RECRUITER FEATURES


recruiters["company_key"] = recruiters["company"].astype(str).str.lower().str.strip()

recruiters["gmail_email_flag"] = recruiters["email"].astype(str).str.lower().str.contains(
    "gmail|yahoo|outlook|hotmail", regex=True
).astype(int)

recruiters["company_email_flag"] = 1 - recruiters["gmail_email_flag"]
recruiters["linkedin_exists"] = recruiters["linkedin_url"].notna().astype(int)
recruiters["verified_flag"] = recruiters["verified"].apply(bool_to_int)

recruiters["previous_reports_value"] = pd.to_numeric(
    recruiters["previous_reports"], errors="coerce"
).fillna(0)

recruiter_features = recruiters.groupby("company_key").agg({
    "gmail_email_flag": "max",
    "company_email_flag": "max",
    "linkedin_exists": "max",
    "verified_flag": "max",
    "previous_reports_value": "max"
}).reset_index()

df = df.merge(recruiter_features, on="company_key", how="left")

for col in [
    "gmail_email_flag", "company_email_flag", "linkedin_exists",
    "verified_flag", "previous_reports_value"
]:
    df[col] = df[col].fillna(0)

df["recruiter_risk"] = (
    df["gmail_email_flag"] * 0.30 +
    (1 - df["linkedin_exists"]) * 0.20 +
    (1 - df["verified_flag"]) * 0.30 +
    (df["previous_reports_value"] > 0).astype(int) * 0.20
)



# 9. SCAM REPORT NLP FEATURES


scam_reports["report_text"] = ""

if "report_reason" in scam_reports.columns:
    scam_reports["report_text"] += scam_reports["report_reason"].apply(clean_text)

if "user_comment" in scam_reports.columns:
    scam_reports["report_text"] += " " + scam_reports["user_comment"].apply(clean_text)

scam_reports["severity_numeric"] = pd.to_numeric(
    scam_reports["severity"], errors="coerce"
).fillna(0)

if "report_id" in df.columns and "job_id" in scam_reports.columns:
    report_features = scam_reports.groupby("job_id").agg(
        report_count=("severity_numeric", "count"),
        avg_severity=("severity_numeric", "mean"),
        max_severity=("severity_numeric", "max"),
        report_text=("report_text", " ".join)
    ).reset_index()

    df = df.merge(report_features, left_on="report_id", right_on="job_id", how="left")
else:
    df["report_count"] = 0
    df["avg_severity"] = 0
    df["max_severity"] = 0
    df["report_text"] = ""

for col in ["report_count", "avg_severity", "max_severity"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

df["report_text"] = df["report_text"].fillna("")
df["combined_text"] = df["combined_text"] + " " + df["report_text"]

max_avg_severity = max(df["avg_severity"].max(), 1)

df["report_risk"] = (
    (df["report_count"] > 0).astype(int) * 0.50 +
    (df["avg_severity"] / max_avg_severity).clip(0, 1) * 0.50
)



# 10. SALARY ANOMALY


salary_median = df["salary_numeric"].median()
salary_std = df["salary_numeric"].std()

if salary_std == 0 or pd.isna(salary_std):
    salary_std = 1

df["salary_zscore"] = abs((df["salary_numeric"] - salary_median) / salary_std)
df["very_high_salary_flag"] = (df["salary_numeric"] > 50000).astype(int)
df["salary_risk"] = ((df["salary_zscore"] > 3) | (df["salary_numeric"] > 50000)).astype(int)



# 11. RULE-BASED SCAM SCORE


df["rule_based_scam_score"] = (
    0.30 * df["keyword_risk"] +
    0.20 * df["domain_risk"] +
    0.15 * df["salary_risk"] +
    0.10 * df["recruiter_risk"] +
    0.10 * df["report_risk"] +
    0.05 * df["domain_impersonation_risk"] +
    0.05 * (df["fraud_phrase_count"] > 0).astype(int) +
    0.05 * (df["urgency_count"] > 0).astype(int)
) * 100

df["rule_based_risk_level"] = df["rule_based_scam_score"].apply(risk_level)



# 12. FINAL FEATURES


numeric_features = [
    "salary_numeric",
    "keyword_score",
    "keyword_matches",
    "keyword_risk",
    "fraud_phrase_count",
    "urgency_count",
    "sentiment_polarity",
    "sentiment_subjectivity",
    "org_entity_count",
    "money_entity_count",
    "person_entity_count",
    "domain_age_days",
    "ssl_valid",
    "trust_score",
    "blacklisted",
    "domain_risk",
    "domain_company_similarity",
    "domain_impersonation_risk",
    "gmail_email_flag",
    "company_email_flag",
    "linkedin_exists",
    "verified_flag",
    "previous_reports_value",
    "recruiter_risk",
    "report_count",
    "avg_severity",
    "max_severity",
    "report_risk",
    "salary_zscore",
    "salary_risk",
    "very_high_salary_flag"
]

X = df[["combined_text"] + numeric_features].copy()
y = df["is_flagged"].astype(int)

model_data = pd.concat([X, y.rename("target"), df["company_key"]], axis=1)

# Leakage reduction: remove duplicate text rows
model_data = model_data.drop_duplicates(subset=["combined_text"])

X = model_data[["combined_text"] + numeric_features]
y = model_data["target"]
groups = model_data["company_key"].fillna("unknown")

print("\nFinal dataset:", X.shape)
print("Target distribution:")
print(y.value_counts())



# 13. LEAKAGE-SAFE SPLIT


if groups.nunique() > 2:
    splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
    train_idx, test_idx = next(splitter.split(X, y, groups))

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        stratify=y,
        random_state=42
    )



# 14. NLP VECTORIZERS


tfidf_vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 3),
    min_df=2,
    max_df=0.95,
    stop_words="english"
)

bow_vectorizer = CountVectorizer(
    max_features=12000,
    ngram_range=(1, 3),
    min_df=2,
    max_df=0.95,
    stop_words="english"
)

preprocessor_tfidf = ColumnTransformer([
    ("tfidf_text", tfidf_vectorizer, "combined_text"),
    ("num", StandardScaler(), numeric_features)
])

preprocessor_bow_only = ColumnTransformer([
    ("bow_text", bow_vectorizer, "combined_text")
])



# 15. TRAIN ALL MODELS


models = {
    "Logistic_Regression_TFIDF": Pipeline([
        ("preprocessor", preprocessor_tfidf),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", n_jobs=-1))
    ]),

    "Naive_Bayes_BOW": Pipeline([
        ("preprocessor", preprocessor_bow_only),
        ("model", MultinomialNB())
    ]),

    "Random_Forest_TFIDF": Pipeline([
        ("preprocessor", preprocessor_tfidf),
        ("model", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ])
}

if XGBOOST_AVAILABLE:
    models["XGBoost_TFIDF"] = Pipeline([
        ("preprocessor", preprocessor_tfidf),
        ("model", XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric="logloss",
            random_state=42
        ))
    ])

os.makedirs("/content/models", exist_ok=True)

results = {}
trained_models = {}

for name, model in models.items():
    print("\n" + "=" * 70)
    print("Training:", name)
    print("=" * 70)

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    try:
        y_prob = model.predict_proba(X_test)[:, 1]
    except:
        y_prob = y_pred

    results[name] = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob)
    }

    trained_models[name] = model

    print(results[name])
    print(classification_report(y_test, y_pred))
    print(confusion_matrix(y_test, y_pred))

    joblib.dump(model, f"/content/models/{name}.pkl")



# 16. SELECT BEST MODEL


results_df = pd.DataFrame(results).T.sort_values("f1_score", ascending=False)

print("\nMODEL COMPARISON:")
print(results_df)

best_model_name = results_df.index[0]
best_model = trained_models[best_model_name]

print("\nBEST MODEL SELECTED:", best_model_name)

results_df.to_csv("/content/models/model_comparison.csv")
joblib.dump(best_model, "/content/models/best_fake_job_scam_model.pkl")



# 17. ENSEMBLE SCORE ALSO


weights = {
    "Logistic_Regression_TFIDF": 0.20,
    "Naive_Bayes_BOW": 0.20,
    "Random_Forest_TFIDF": 0.25,
    "XGBoost_TFIDF": 0.35
}

ensemble_prob = np.zeros(len(X_test))
total_weight = 0

for name, model in trained_models.items():
    w = weights.get(name, 0.25)

    try:
        prob = model.predict_proba(X_test)[:, 1]
    except:
        prob = model.predict(X_test)

    ensemble_prob += w * prob
    total_weight += w

ensemble_prob = ensemble_prob / total_weight
ensemble_pred = (ensemble_prob >= 0.5).astype(int)

ensemble_result = {
    "accuracy": accuracy_score(y_test, ensemble_pred),
    "precision": precision_score(y_test, ensemble_pred, zero_division=0),
    "recall": recall_score(y_test, ensemble_pred, zero_division=0),
    "f1_score": f1_score(y_test, ensemble_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, ensemble_prob)
}

print("\nENSEMBLE RESULT:")
print(ensemble_result)


# 18. SAVE FINAL PREDICTIONS


final_output = X_test.copy()
final_output["actual_label"] = y_test.values
final_output["ml_scam_probability"] = best_model.predict_proba(X_test)[:, 1]
final_output["ml_scam_score"] = final_output["ml_scam_probability"] * 100
final_output["ml_risk_level"] = final_output["ml_scam_score"].apply(risk_level)

final_output.to_csv("/content/models/final_predictions.csv", index=False)

joblib.dump(trained_models, "/content/models/all_trained_models.pkl")
joblib.dump(keyword_dict, "/content/models/keyword_dictionary.pkl")

print("\nTraining completed.")
print("Best model saved at: /content/models/best_fake_job_scam_model.pkl")
print("All outputs saved inside: /content/models/")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 37.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
All files found.
(19261, 11) (4000, 8) (4000, 6) (4000, 6) (4000, 3)
Applying spaCy lemmatization...

Final dataset: (9077, 32)
Target distribution:
target
0    5979
1    3098
Name: count, dtype: int64

Training: Logistic_Regression_TFIDF
{'accuracy': 0.926056338028169, 'precision': 0.8487518355359766, 'recall': 0.961730449251248, 'f1_score': 0.9017160686427457, 'roc_auc': np.float64(0.981016830516682)}
              precision    recall  f1-score   support

           0       0.98      0.91      0.94      1103
           1       0.85      0.96      0.90       601

    accuracy